# 04 · Connecting the application to data and tools

**Session 01 · notebook 4 of 6**

## Objective

Give the model a tool that returns authoritative statistics, watch it call the tool, make its output a typed object that other software can read, and expose the same tool over the Model Context Protocol with FastMCP.

## The application after this notebook

```
question ──> model ──┬──> tool call: get_indicator(indicator, country, years) ──> data ──┐
                     │<──────────────────────────────────────────────────────────────────┘
                     └──> answer, or an object that matches a schema

the same tool, served over MCP ──> any MCP client (this notebook, a chatbot, a desktop assistant)
```

Passages are for definitions and context. Numbers come from a tool over the data.

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/worldbank/ai4data.git"   # replace with the published repository when released
BRANCH = "feat/training-series"
SESSION = "training/session-01-building-ai-applications-v1/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook replays recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
model = w.make_model()
print("Model:", w.mode(model))

Model: recorded outputs (no WORKSHOP_API_KEY set)


## 1. A number from memory and a number from data

In [2]:
w.use("nb04_from_memory")
w.show(model.invoke("What was Rwanda's GDP per capita in 2023?"))
print()
r = w.indicator_lookup("NY.GDP.PCAP.CD", "Rwanda", start=2023, end=2023)
print("Data:", r["name"], r["observations"], "·", r["source"])

[recorded output: nb04_from_memory]
Rwanda's GDP per capita in 2023 was approximately 1,000 US dollars (current prices), according to
World Bank data. Please verify against the latest World Development Indicators, as figures are
revised.

Data: GDP per capita (current US$) [{'year': 2023, 'value': 1061.0579, 'claim_id': 'NY.GDP.PCAP.CD:RWA:2023'}] · World Development Indicators (Country official statistics)


The memory answer is close and hedged, with no way for a reader to tell how close. The data answer carries the indicator, the year, the value, the source, and a `claim_id` that notebook 06 will use to verify it.

## 2. The tool and its schema

`get_indicator` is a Python function decorated with LangChain's `@tool`. The docstring becomes the description the model reads, and the type hints become the parameter schema. The model fills the parameters from the question.

In [3]:
import json

print("name:       ", w.get_indicator.name)
print("description:", w.get_indicator.description)
print(json.dumps(w.get_indicator.args, indent=2))
print()
print("available indicators:")
for i in w.list_indicators():
    print(f"  {i['id']:20s} {i['name']}")

name:        get_indicator
description: Observations of a World Development Indicators series for one country. Use it for
    any statistic; do not quote figures from memory. `indicator` is an id such as
    IT.NET.USER.ZS or NY.GDP.PCAP.CD; `country` is a name or ISO3 code; `start` and `end`
    bound the years.
{
  "indicator": {
    "title": "Indicator",
    "type": "string"
  },
  "country": {
    "title": "Country",
    "type": "string"
  },
  "start": {
    "anyOf": [
      {
        "type": "integer"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "title": "Start"
  },
  "end": {
    "anyOf": [
      {
        "type": "integer"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "title": "End"
  }
}

available indicators:
  NY.GDP.PCAP.CD       GDP per capita (current US$)
  IT.NET.USER.ZS       Individuals using the Internet (% of population)
  NY.GDP.MKTP.KD.ZG    GDP growth (annual %)
  SP.POP.TOTL          Populatio

## 3. The model calls the tool, step by step

`bind_tools` tells the model which tools exist. The model answers with a tool call; the application runs the tool, returns the result as a `ToolMessage`, and calls the model again. Notebook 05 lets LangGraph run this loop; here it is written out once so that the mechanics are visible.

In [4]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

INSTRUCTIONS = """You are a research assistant for development data.
Use get_indicator for every statistic; do not quote figures from memory. Name the indicator and the source with each figure.
If the tool returns no observation for the year asked, say so and give the latest available year."""

model_with_tools = model.bind_tools([w.get_indicator])
messages = [SystemMessage(content=INSTRUCTIONS), HumanMessage(content="What was Rwanda's GDP per capita in 2023?")]

w.use("nb04_tool_call")
reply = model_with_tools.invoke(messages)
w.show(reply)
for call in reply.tool_calls:
    result = w.get_indicator.invoke(call["args"])
    print("tool result:", json.dumps(result)[:160], "…")
    messages += [reply, ToolMessage(content=json.dumps(result), tool_call_id=call["id"])]
print()
w.show(model_with_tools.invoke(messages))

[recorded output: nb04_tool_call]
-> tool call get_indicator({"indicator": "NY.GDP.PCAP.CD", "country": "Rwanda", "start": 2020, "end": 2024})

tool result: {"indicator": "NY.GDP.PCAP.CD", "country": "RWA", "name": "GDP per capita (current US$)", "source": "World Development Indicators (Country official statistics)" …

[recorded output: nb04_tool_call]
Rwanda's GDP per capita in 2023 was 1,061 US dollars in current prices (World Development
Indicators, NY.GDP.PCAP.CD). It rose from 803 dollars in 2020 to 1,007 in 2022 and 1,061 in 2023,
and was 1,060 in 2024.


## 4. Data that does not exist

The planted problem: a year with no observation. The tool returns an empty list, and the instructions say what to do with it.

In [5]:
messages = [SystemMessage(content=INSTRUCTIONS), HumanMessage(content="What share of Kenya's population used the internet in 2025?")]
w.use("nb04_missing_year")
reply = model_with_tools.invoke(messages)
for call in reply.tool_calls:
    result = w.get_indicator.invoke(call["args"])
    print("tool result for 2025:", result["observations"])
    messages += [reply, ToolMessage(content=json.dumps(result), tool_call_id=call["id"])]
print()
w.show(model_with_tools.invoke(messages))

tool result for 2025: []

[recorded output: nb04_missing_year]
No observation is available for 2025. The series IT.NET.USER.ZS (Individuals using the Internet, %
of population) has no 2025 value for Kenya in the data returned. The latest available year is 2024,
when the value was 35.0 percent.


Compare with notebook 01, where the model answered the same question from a brief. Here the absence is a fact returned by the tool, which the application can test for.

## 5. Structured output

An answer that another program will read, store, or display needs a fixed shape. `ComparisonAnswer` is a Pydantic model; `with_structured_output` sends its schema with the request and validates the response on the way back.

In [6]:
print(json.dumps(w.ComparisonAnswer.model_json_schema(), indent=2)[:1200], "…")

{
  "$defs": {
    "CountrySummary": {
      "properties": {
        "country": {
          "title": "Country",
          "type": "string"
        },
        "internet_latest": {
          "$ref": "#/$defs/Observation"
        },
        "gdp_per_capita_latest": {
          "$ref": "#/$defs/Observation"
        },
        "trend": {
          "description": "One or two sentences on the trend since 2010",
          "title": "Trend",
          "type": "string"
        }
      },
      "required": [
        "country",
        "internet_latest",
        "gdp_per_capita_latest",
        "trend"
      ],
      "title": "CountrySummary",
      "type": "object"
    },
    "Observation": {
      "properties": {
        "indicator": {
          "description": "Indicator id",
          "title": "Indicator",
          "type": "string"
        },
        "country": {
          "description": "ISO3 code",
          "title": "Country",
          "type": "string"
        },
        "year": {
         

In [7]:
data = [w.indicator_lookup(i, c, start=2024) for i in ["IT.NET.USER.ZS", "NY.GDP.PCAP.CD"] for c in ["Kenya", "Rwanda"]]
structured = model.with_structured_output(w.ComparisonAnswer)

w.use("nb04_structured")
parsed = structured.invoke([
    SystemMessage(content="Fill the schema from the data you are given. Use the claim_id values exactly as given."),
    HumanMessage(content="DATA:\n" + json.dumps(data) + "\n\nQUESTION: " + w.MAIN_QUESTION),
])
for cs in parsed.countries:
    print(cs.country, "internet", cs.internet_latest.year, cs.internet_latest.value, "| GDP pc", cs.gdp_per_capita_latest.year, round(cs.gdp_per_capita_latest.value))
print()
print(parsed.comparison)
print(parsed.sources)

KEN internet 2024 34.9762 | GDP pc 2024 2133
RWA internet 2024 31.7033 | GDP pc 2024 1060

Kenya has the higher level on both indicators and Rwanda the faster growth on both. Internet use in the two countries converged to about a third of the population by 2024, while GDP per capita in Kenya remained about twice Rwanda's.
['World Development Indicators, IT.NET.USER.ZS (ITU)', 'World Development Indicators, NY.GDP.PCAP.CD (World Bank national accounts)']


Validation is what makes the schema a contract. The cell below damages the JSON and shows the error that a downstream program would receive. Without validation it would receive a wrong value and no error.

In [8]:
from pydantic import ValidationError

damaged = parsed.model_dump_json().replace('"year":2024', '"year":"twenty twenty-four"', 1)
try:
    w.ComparisonAnswer.model_validate_json(damaged)
except ValidationError as e:
    print(e.errors()[0]["loc"], "->", e.errors()[0]["msg"])

('countries', 0, 'internet_latest', 'year') -> Input should be a valid integer, unable to parse string as an integer


## 6. The same tool over MCP

The Model Context Protocol (MCP) is a standard way to expose tools to any client. `indicator_server.py` wraps `get_indicator` in a FastMCP server in a dozen lines; the FastMCP client starts it as a subprocess, lists its tools, and calls one. A chatbot, a desktop assistant, or an agent connects the same way.

In [9]:
from fastmcp import Client

async with Client(w.mcp_server_transport()) as client:
    for t in await client.list_tools():
        print(f"{t.name:16s} {t.description[:90]}")
    result = await client.call_tool("get_indicator", {"indicator": "IT.NET.USER.ZS", "country": "Rwanda", "start": 2022})
    print()
    print(result.data)

list_indicators  The indicators this server holds, with their ids and names.
get_indicator    Observations of a World Development Indicators series for one country. `indicator`
is an i

{'indicator': 'IT.NET.USER.ZS', 'country': 'RWA', 'name': 'Individuals using the Internet (% of population)', 'source': 'World Development Indicators (World Telecommunication/ICT Indicators Database)', 'observations': [{'year': 2022, 'value': 28.0168, 'claim_id': 'IT.NET.USER.ZS:RWA:2022'}, {'year': 2023, 'value': 28.8058, 'claim_id': 'IT.NET.USER.ZS:RWA:2023'}, {'year': 2024, 'value': 31.7033, 'claim_id': 'IT.NET.USER.ZS:RWA:2024'}]}


A LangChain tool can wrap the MCP call, so an agent uses the server like any other tool. The cell defines one and runs an agent over it (the agent itself is the subject of notebook 05).

In [10]:
from langchain_core.tools import tool
from langchain.agents import create_agent


@tool("get_indicator")
async def get_indicator_over_mcp(indicator: str, country: str, start: int | None = None, end: int | None = None) -> dict:
    """Observations of a World Development Indicators series for one country, from the indicator MCP server."""
    async with Client(w.mcp_server_transport()) as client:
        args = {k: v for k, v in dict(indicator=indicator, country=country, start=start, end=end).items() if v is not None}
        return (await client.call_tool("get_indicator", args)).data


agent = create_agent(model, [get_indicator_over_mcp], system_prompt=INSTRUCTIONS)
w.use("nb04_mcp")
result = await agent.ainvoke({"messages": [HumanMessage(content="How has internet use in Rwanda changed since 2022?")]})
w.print_trace(result)
print()
w.show(w.final_text(result))

step 1: get_indicator({"indicator": "IT.NET.USER.ZS", "country": "Rwanda", "start": 2022, "end": 2024})
   -> {"indicator": "IT.NET.USER.ZS", "country": "RWA", "name": "Individuals using the Internet (% of population)", "source": "World Development Indicators …

Rwanda's share of individuals using the internet was 28.0 percent in 2022, 28.8 percent in 2023, and
31.7 percent in 2024 (World Development Indicators, IT.NET.USER.ZS, through the indicator MCP
server).


The program's [Data360 MCP server](https://github.com/worldbank/data360-mcp) exposes search, metadata, and data tools over the World Bank's data platform in the same way, at a public endpoint. The cell lists its tools and runs one search; it is guarded because the session must not depend on the network.

In [11]:
try:
    async with Client(w.DATA360_MCP_URL) as client:
        tools = await client.list_tools()
        print(len(tools), "tools, for example:", [t.name for t in tools][:5])
        found = await client.call_tool("data360_search_indicators", {"query": "individuals using the internet", "limit": 3})
        for ind in found.data["indicators"]:
            print(f"  {ind['idno']:28s} {ind['name']}")
except Exception as e:
    print("Data360 MCP call skipped:", type(e).__name__)

16 tools, for example: ['data360_search_indicators', 'data360_search_datasets', 'data360_get_metadata', 'data360_get_data', 'data360_get_disaggregation']


  WEF_GCIHH_NETUSERPCT         Individuals using Internet, %
  WB_WDI_IT_NET_USER_ZS        Individuals using the Internet (% of population)
  WEF_TTDI_NETUSERPCT          Individuals using Internet, % of pop


## Key points

- Numbers come from a tool over authoritative data; the model's memory is for language.
- A tool is a function with a description the model can read. The model fills the parameters; the application runs the function and returns the result.
- Absence of data is a tool result the application can test for.
- A schema turns the answer into a contract: validated on the way in, readable by other software on the way out.
- MCP exposes the same tool to any client; FastMCP makes a server out of a function.

## Exercise

Add a `get_population` tool to `indicator_server.py` (series `SP.POP.TOTL`) and ask the agent "How many people in Rwanda used the internet in 2024?". The answer needs two tool calls and a calculation; check whether the model does the arithmetic or asks for a tool to do it.

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows.
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.